# day-32-package-and-iac — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt.

### Exercise 1 — layer size + cache hit-rate

In [1]:
import hashlib
def h(*p): return hashlib.sha256("|".join(map(str,p)).encode()).hexdigest()[:12]
SIZE = {"FROM": 120, "RUN": 200, "COPY_src": 5, "COPY_reqs": 1, "CMD": 0}

def build(instr, ctx, cache):
    chain, layers, hits = "scratch", [], 0
    for op, arg in instr:
        dep = ctx.get(arg, "MISSING") if op == "COPY" else arg
        key = h(chain, op, dep); chain = key
        if key in cache: hits += 1
        sz = SIZE["RUN"] if op == "RUN" else SIZE["FROM"] if op == "FROM" else \
             SIZE["COPY_src"] if arg == "src/" else SIZE["COPY_reqs"] if op == "COPY" else 0
        layers.append((op, arg, key, sz))
    return layers, {l[2] for l in layers}, hits, sum(l[3] for l in layers)

bad  = [("FROM","py"),("COPY","src/"),("RUN","pip"),("CMD","run")]
good = [("FROM","py"),("COPY","requirements.txt"),("RUN","pip"),("COPY","src/"),("CMD","run")]
cb = cg = set()
for i in range(5):
    ctx = {"requirements.txt": h("reqs-A"), "src/": h(f"code-{i}")}
    lb, cb, hb, sb = build(bad, ctx, cb)
    lg, cg, hg, sg = build(good, ctx, cg)
    print(f"commit {i}: bad hits {hb}/{len(lb)}  good hits {hg}/{len(lg)}  (size bad={sb} good={sg})")
print("\nbad order re-runs pip install (200) every commit; good order caches it after commit 0")

commit 0: bad hits 0/4  good hits 0/5  (size bad=325 good=326)
commit 1: bad hits 1/4  good hits 3/5  (size bad=325 good=326)
commit 2: bad hits 1/4  good hits 3/5  (size bad=325 good=326)
commit 3: bad hits 1/4  good hits 3/5  (size bad=325 good=326)
commit 4: bad hits 1/4  good hits 3/5  (size bad=325 good=326)

bad order re-runs pip install (200) every commit; good order caches it after commit 0


### Exercise 2 — `destroy_all` in reverse dependency order

In [2]:
from dataclasses import dataclass, field
@dataclass
class Resource:
    type: str; name: str; config: dict; depends_on: list = field(default_factory=list)
    @property
    def id(self): return f"{self.type}.{self.name}"

def topo(resources):
    rs = {r.id: r for r in resources}; done=set(); out=[]
    def visit(r):
        if r.id in done: return
        for dq in r.depends_on: visit(rs[dq])
        done.add(r.id); out.append(r)
    for r in resources: visit(r)
    return out

repo = Resource("ecr_repo","api",{})
fn   = Resource("lambda","api",{}, ["ecr_repo.api"])
gw   = Resource("api_gateway","api",{}, ["lambda.api"])
stack = [repo, fn, gw]
print("create order :", [r.id for r in topo(stack)])
print("destroy order:", [r.id for r in reversed(topo(stack))])  # gateway -> lambda -> repo

create order : ['ecr_repo.api', 'lambda.api', 'api_gateway.api']
destroy order: ['api_gateway.api', 'lambda.api', 'ecr_repo.api']


### Exercises 3–6 — sketches

**3. `-target`.** `plan(resources, target="lambda.api")`: compute the transitive closure of
`target` over `depends_on`, then run the normal diff but only emit actions whose `rid` is in
that closure. Fixing the §3 drift with `-target lambda.api` leaves `api_gateway` untouched.

**4. Dockerfile review of `instr_bad`.** Changes: (a) put `COPY requirements.txt` + `RUN pip
install` before `COPY src/`; (b) pin the base (`python:3.12-slim`, ideally `@sha256:…`);
(c) split into build + runtime stages; (d) add `useradd` + `USER appuser`; (e) add
`HEALTHCHECK`; (f) add `.dockerignore` for `.env`, `.git`, `tests`, `*.ipynb`; (g) `ENV
PYTHONUNBUFFERED=1` so logs stream. Each either speeds builds, shrinks the image, or reduces
blast radius.

**5. Secret leak.** `ENV ANTHROPIC_API_KEY=sk-ant-…` bakes the key into: the image layer, the
image *history* / config JSON (visible via `docker history` / `inspect`), every registry the
image is pushed to, any CI log that printed the build, and every developer who pulls it.
Remediation: **rotate the key immediately** (assume compromised), remove the `ENV` line, pass
the secret at runtime from a vault, delete affected image tags from the registry, and scan
history so an old tag isn't still pullable.

**6. Terraform module.** `modules/rag_service/` holds the `aws_ecr_repository`,
`aws_lambda_function`, `aws_apigatewayv2_*` resources with `variable "image_tag"`,
`"memory"`, `"model"`, `"env_name"`. Root `staging/main.tf` and `prod/main.tf` each become
`module "rag" { source = "../modules/rag_service"; image_tag = var.sha; memory = 1024; model
= "claude-haiku-4-5"; env_name = "staging" }`. The resource wiring stays in the module; the
knobs that differ per environment become inputs; state stays per-environment.

### Answer key

1. A layer is one build instruction's filesystem diff, content-addressed by hash. The cache
   reuses it only if that instruction *and every earlier layer* are byte-identical to a prior
   build.
2. `requirements.txt` changes rarely; `src/` changes every commit. Copying reqs first means
   the slow `pip install` layer stays cached across code-only changes.
3. **Desired** = your config. **Recorded** = the state file (what the tool believes it
   created). **Real** = the actual cloud. `plan` compares desired vs recorded; **drift** is
   recorded vs real.
4. `terraform apply` is declarative and idempotent — it computes the diff to the desired end
   state, so no diff ⇒ no action. A bash script is a list of steps whose effect depends on the
   starting state, so re-running can double-apply or silently skip.
5. Secrets (→ secrets manager, injected at runtime), environment config like model name / URLs
   (→ env vars set per environment), and large data / build caches / `.git` (→ `.dockerignore`,
   fetched or mounted at runtime). Any three.
6. A multi-stage build keeps compilers, headers, and package caches in an early stage that
   never ships — smaller final image, faster pulls and cold starts, less attack surface.
7. Stop accepting new requests (so the load balancer drains it), let in-flight LLM calls
   finish within the grace period (~30 s), flush logs/traces, then exit. Don't drop live
   streams.